# Market Intelligence Engine — Prototype 3
**Pipeline:** Rule-Based + ML Terarah | **Split:** 60% Train / 40% Test  
**P0 Modules:** Peer Analysis · What Changed · Anomaly Detection · Fundamental Divergence · Opportunity Signal · Risk Signal · Evidence Generation

## 1. Import Library

In [17]:
import os, json, warnings
from datetime import datetime

import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import IsolationForest, RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', '{:.4f}'.format)

# ── Unified Data API (local import)
import sys, pathlib
NB_DIR = pathlib.Path(globals().get('__vsc_ipynb_file__', __file__) if '__file__' in dir() else os.getcwd()).parent
REPO_ROOT = NB_DIR if (NB_DIR / 'unified_data').exists() else NB_DIR.parent
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))
from unified_data.endpoint_finaldata import get_api_data

# ── Konstanta
RANDOM_STATE = 42
TRAIN_RATIO  = 0.60          # 60% train / 40% test

# ── Daftar ticker yang akan dianalisis
TICKERS = ['BBCA', 'BBRI', 'BMRI', 'TLKM', 'ANTM', 'BUMI']

print('Unified Data API loaded.')
print(f'Tickers   : {TICKERS}')
print(f'Train/Test: {TRAIN_RATIO:.0%} / {1-TRAIN_RATIO:.0%}')


Unified Data API loaded.
Tickers   : ['BBCA', 'BBRI', 'BMRI', 'TLKM', 'ANTM', 'BUMI']
Train/Test: 60% / 40%


## 2. Data Loading

In [18]:
# ── 2a. Fetch data harga 5 tahun via Unified Data API
frames = []
for ticker in TICKERS:
    res = get_api_data(ticker=ticker, data_type='price', period='10y')
    if res['status'] == 'SUCCESS':
        data = res['data']
        records = data.get('records', [])
        if not records:
            print(f'  SKIP {ticker}: empty records')
            continue
        tmp = pd.DataFrame(records)
        tmp['Ticker'] = ticker
        # Pastikan kolom numerik ada (adj_close, dividends, stock_splits sebagai default 0)
        tmp['adj_close']    = tmp.get('adj_close',    tmp['close'])
        tmp['dividends']    = tmp.get('dividends',    0.0)
        tmp['stock_splits'] = tmp.get('stock_splits', 0.0)
        frames.append(tmp)
        print(f'  Loaded {ticker}: {len(tmp):,} rows  ({data.get("date_range", "")})')
    else:
        print(f'  SKIP {ticker}: {res.get("status")} — {res.get("data", {}).get("error", "")}')

df_raw = pd.concat(frames, ignore_index=True)
print(f'\nTotal combined rows: {len(df_raw):,}')
df_raw.head(3)


  Loaded BBCA: 2,464 rows  ()
  Loaded BBRI: 2,465 rows  ()
  Loaded BMRI: 2,465 rows  ()
  Loaded TLKM: 2,464 rows  ()
  Loaded ANTM: 2,465 rows  ()
  Loaded BUMI: 2,465 rows  ()

Total combined rows: 14,788


,date,open,high,low,close,volume,Ticker,adj_close,dividends,stock_splits
0,2016-09-14,2371.4696,2379.4010,2347.6756,2375.4353,92904000,BBCA,2375.4353,0.0000,0.0000
1,2016-09-15,2411.1259,2426.9885,2375.4348,2395.2632,159713000,BBCA,2395.2632,0.0000,0.0000
2,2016-09-16,2395.2632,2415.0915,2379.4005,2395.2632,151477000,BBCA,2395.2632,0.0000,0.0000


## 3. Data Cleaning & Feature Engineering

In [19]:
# 3a. Type conversion
df = df_raw.copy()
df['date'] = pd.to_datetime(df['date'], errors='coerce')

NUM_COLS = ['open','high','low','close','adj_close','volume','dividends','stock_splits']
for c in NUM_COLS:
    df[c] = pd.to_numeric(df[c], errors='coerce')

# 3b. Deduplicate
before = len(df)
df.drop_duplicates(subset=['date','Ticker'], keep='last', inplace=True)
print(f'Duplicates removed: {before - len(df)}')

# 3c. Drop bad dates
df.dropna(subset=['date'], inplace=True)

# 3d. Last 5 years only
cutoff = pd.Timestamp.today() - pd.DateOffset(years=10)
df = df[df['date'] >= cutoff].copy()

# 3e. Remove inf
df.replace([np.inf, -np.inf], np.nan, inplace=True)

# 3f. Forward/backward fill per ticker
df.sort_values(['Ticker','date'], inplace=True)
df[NUM_COLS] = df.groupby('Ticker')[NUM_COLS].transform(
    lambda x: x.fillna(method='ffill').fillna(method='bfill')
)
df.dropna(subset=['close'], inplace=True)

print(f'Rows after cleaning (10-year window): {len(df):,}')
print('Tickers:', sorted(df['Ticker'].unique().tolist()))


Duplicates removed: 0
Rows after cleaning (10-year window): 14,782
Tickers: ['ANTM', 'BBCA', 'BBRI', 'BMRI', 'BUMI', 'TLKM']


## 3b. Fetch Real Fundamental Data (Valuation, Forecast, Dividend, Institutional)


In [20]:
# ── Fetch fundamental nyata per ticker dari Unified Data API
fundamental = {}
for ticker in TICKERS:
    print(f'Fetching {ticker}...', end=' ', flush=True)
    val  = get_api_data(ticker=ticker, data_type='valuation',                period='current')
    fc   = get_api_data(ticker=ticker, data_type='forecast',                 period='current')
    div  = get_api_data(ticker=ticker, data_type='dividend',                 period='10y')
    inst = get_api_data(ticker=ticker, data_type='institutional_transactions',period='6m')

    fundamental[ticker] = {
        # Valuation — untuk margin of safety
        'forward_pe'      : val['data'].get('forward_pe')       if val['status']=='SUCCESS' else None,
        'trailing_pe'     : val['data'].get('trailing_pe')      if val['status']=='SUCCESS' else None,
        'intrinsic_value' : val['data'].get('intrinsic_value')  if val['status']=='SUCCESS' else None,
        'price_to_book'   : val['data'].get('price_to_book')    if val['status']=='SUCCESS' else None,
        # Forecast — target analis & pertumbuhan EPS
        'target_mean'     : fc['data'].get('target_mean_price') if fc['status']=='SUCCESS' else None,
        'target_high'     : fc['data'].get('target_high_price') if fc['status']=='SUCCESS' else None,
        'eps_growth'      : fc['data'].get('eps_growth')        if fc['status']=='SUCCESS' else None,
        'revenue_growth'  : fc['data'].get('revenue_growth')    if fc['status']=='SUCCESS' else None,
        # Dividend
        'dividend_yield'  : div['data'].get('dividend_yield')   if div['status']=='SUCCESS' else None,
        'payout_ratio'    : div['data'].get('payout_ratio')     if div['status']=='SUCCESS' else None,
        # Institutional
        'inst_top_pct'    : inst['data'].get('top_holder_pct_held') if inst['status']=='SUCCESS' else None,
    }
    print('OK')

# Ringkasan
print('\nFundamental snapshot:')
pd.DataFrame(fundamental).T


Fetching BBCA... OK
Fetching BBRI... OK
Fetching BMRI... OK
Fetching TLKM... OK
Fetching ANTM... OK
Fetching BUMI... OK

Fundamental snapshot:


,forward_pe,trailing_pe,intrinsic_value,price_to_book,target_mean,target_high,eps_growth,revenue_growth,dividend_yield,payout_ratio,inst_top_pct
BBCA,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,6.0200,0.7542,NaN
BBRI,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,12.7800,0.8501,NaN
BMRI,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,12.4500,0.7147,NaN
TLKM,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,8.5800,1.2516,NaN
ANTM,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,6.4200,0.5669,NaN
BUMI,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0000,NaN


In [21]:
# 3g. Feature Engineering

# Returns
df['return_1d']  = df.groupby('Ticker')['close'].pct_change(1)
df['return_5d']  = df.groupby('Ticker')['close'].pct_change(5)
df['return_20d'] = df.groupby('Ticker')['close'].pct_change(20)
df['return_60d'] = df.groupby('Ticker')['close'].pct_change(60)

# Volatility
df['vol_20d'] = df.groupby('Ticker')['return_1d'].transform(lambda x: x.rolling(20).std())
df['vol_60d'] = df.groupby('Ticker')['return_1d'].transform(lambda x: x.rolling(60).std())

# Volume momentum
df['vol_avg_20d'] = df.groupby('Ticker')['volume'].transform(lambda x: x.rolling(20).mean())
df['vol_ratio']   = df['volume'] / (df['vol_avg_20d'] + 1e-9)

# Intraday range
df['intraday_range'] = (df['high'] - df['low']) / (df['close'] + 1e-9)

# Moving averages & price position
df['ma_20']  = df.groupby('Ticker')['close'].transform(lambda x: x.rolling(20).mean())
df['ma_60']  = df.groupby('Ticker')['close'].transform(lambda x: x.rolling(60).mean())
df['ma_200'] = df.groupby('Ticker')['close'].transform(lambda x: x.rolling(200).mean())
df['price_vs_ma20']  = (df['close'] - df['ma_20'])  / (df['ma_20']  + 1e-9)
df['price_vs_ma60']  = (df['close'] - df['ma_60'])  / (df['ma_60']  + 1e-9)
df['price_vs_ma200'] = (df['close'] - df['ma_200']) / (df['ma_200'] + 1e-9)

# RSI-14
def compute_rsi(s, period=14):
    delta = s.diff()
    gain  = delta.clip(lower=0).rolling(period).mean()
    loss  = (-delta.clip(upper=0)).rolling(period).mean()
    return 100 - (100 / (1 + gain / (loss + 1e-9)))

df['rsi_14'] = df.groupby('Ticker')['close'].transform(compute_rsi)

# Proxy features
df['div_yield']          = df['dividends'] / (df['close'] + 1e-9)
df['valuation_proxy']    = -df['return_60d']            # contrarian: low momentum = cheaper
df['growth_proxy']       = df['return_20d']
df['institutional_flow'] = np.where(df['vol_ratio'] > 2.0, df['vol_ratio'], 0.0)
df['forecast_proxy']     = df['return_5d'] - df['return_20d']

FEATURE_COLS = [
    'return_1d','return_5d','return_20d','return_60d',
    'vol_20d','vol_60d','vol_ratio','intraday_range',
    'price_vs_ma20','price_vs_ma60','price_vs_ma200',
    'rsi_14','div_yield','valuation_proxy','growth_proxy',
    'institutional_flow','forecast_proxy'
]

df.dropna(subset=FEATURE_COLS, inplace=True)
df.reset_index(drop=True, inplace=True)
print(f'Rows after feature engineering: {len(df):,}')
df[FEATURE_COLS].describe()


Rows after feature engineering: 13,588


,return_1d,return_5d,return_20d,return_60d,vol_20d,vol_60d,vol_ratio,intraday_range,price_vs_ma20,price_vs_ma60,price_vs_ma200,rsi_14,div_yield,valuation_proxy,growth_proxy,institutional_flow,forecast_proxy
count,13588.0000,13588.0000,13588.0000,13588.0000,13588.0000,13588.0000,13588.0000,13588.0000,13588.0000,13588.0000,13588.0000,13588.0000,13588.0000,13588.0000,13588.0000,13588.0000,13588.0000
mean,0.0006,0.0030,0.0119,0.0369,0.0223,0.0230,1.0273,0.0298,0.0034,0.0105,0.0367,50.4229,0.0000,-0.0369,0.0119,0.1855,-0.0089
std,0.0261,0.0586,0.1226,0.2520,0.0134,0.0122,0.7155,0.0237,0.0645,0.1215,0.2274,16.7873,0.0000,0.2520,0.1226,0.7884,0.1059
min,-0.1497,-0.3465,-0.5087,-0.5857,0.0000,0.0000,0.0000,0.0000,-0.4206,-0.4968,-0.5916,0.0000,0.0000,-3.0784,-0.5087,0.0000,-0.9700
25%,-0.0116,-0.0236,-0.0458,-0.0807,0.0137,0.0145,0.6404,0.0155,-0.0256,-0.0480,-0.0817,39.5349,0.0000,-0.1078,-0.0458,0.0000,-0.0469
50%,0.0000,0.0000,0.0028,0.0129,0.0190,0.0196,0.8875,0.0237,0.0012,0.0041,0.0247,50.7042,0.0000,-0.0129,0.0028,0.0000,-0.0027
75%,0.0107,0.0252,0.0550,0.1078,0.0262,0.0279,1.2345,0.0368,0.0289,0.0568,0.1190,61.9048,0.0000,0.0807,0.0550,0.0000,0.0396
max,0.3377,0.6456,1.2366,3.0784,0.1055,0.0853,18.7990,0.3147,0.5653,1.1394,2.5742,100.0000,0.0000,0.5857,1.2366,18.7990,0.5798


## 4. Modeling & Execution (Rule-Based + ML Terarah)

### 4.0  Train / Test Split — Temporal (60% / 40%)

In [22]:
# Label: next-day direction  (1=up, 0=down/flat)
df.sort_values(['Ticker','date'], inplace=True)
df['label'] = (df.groupby('Ticker')['close'].shift(-1) > df['close']).astype(int)
df.dropna(subset=['label'], inplace=True)

# Temporal split — respect time order
unique_dates = sorted(df['date'].unique())
split_idx    = int(len(unique_dates) * TRAIN_RATIO)
split_date   = unique_dates[split_idx]

train_df = df[df['date'] <  split_date].copy()
test_df  = df[df['date'] >= split_date].copy()

X_train, y_train = train_df[FEATURE_COLS], train_df['label']
X_test,  y_test  = test_df[FEATURE_COLS],  test_df['label']

print(f"Train period : {train_df['date'].min().date()} -> {train_df['date'].max().date()}  ({len(train_df):,} rows)")
print(f"Test  period : {test_df['date'].min().date()}  -> {test_df['date'].max().date()}   ({len(test_df):,} rows)")
print(f"Train label dist:\n{y_train.value_counts(normalize=True).round(3)}")
print(f"Test  label dist:\n{y_test.value_counts(normalize=True).round(3)}")


Train period : 2017-07-03 -> 2022-11-25  (8,153 rows)
Test  period : 2022-11-28  -> 2026-09-14   (5,435 rows)
Train label dist:
label
0   0.5900
1   0.4100
Name: proportion, dtype: float64
Test  label dist:
label
0   0.5580
1   0.4420
Name: proportion, dtype: float64


### 4.1  Feature Scaling (StandardScaler — fit on train only)

In [23]:
scaler     = StandardScaler()
X_train_sc = scaler.fit_transform(X_train)
X_test_sc  = scaler.transform(X_test)
print('Scaling complete. Mean~0 / Std~1 verified on training set.')


Scaling complete. Mean~0 / Std~1 verified on training set.


### 4.2  Model Training & Accuracy Evaluation (60/40 Split)

In [24]:
MODELS = {
    'Logistic Regression': LogisticRegression(max_iter=1000, random_state=RANDOM_STATE),
    'Random Forest'      : RandomForestClassifier(n_estimators=200, max_depth=8,
                                                   min_samples_leaf=10, random_state=RANDOM_STATE),
}

results = {}
for name, model in MODELS.items():
    model.fit(X_train_sc, y_train)
    y_pred = model.predict(X_test_sc)
    acc    = accuracy_score(y_test, y_pred)
    results[name] = {'model': model, 'acc': acc, 'y_pred': y_pred}
    print(f"\n{'='*55}")
    print(f"Model         : {name}")
    print(f"Test Accuracy (40% hold-out): {acc:.4f}  ({acc*100:.2f}%)")
    print(classification_report(y_test, y_pred, target_names=['Down/Flat', 'Up']))
    print('Confusion Matrix:')
    print(confusion_matrix(y_test, y_pred))

best_name  = max(results, key=lambda k: results[k]['acc'])
best_model = results[best_name]['model']
print(f"\n✅ Best Model Selected : {best_name}  (Acc = {results[best_name]['acc']:.4f})")



Model         : Logistic Regression
Test Accuracy (40% hold-out): 0.5592  (55.92%)
              precision    recall  f1-score   support

   Down/Flat       0.56      0.99      0.71      3031
          Up       0.56      0.02      0.03      2404

    accuracy                           0.56      5435
   macro avg       0.56      0.50      0.37      5435
weighted avg       0.56      0.56      0.41      5435

Confusion Matrix:
[[3002   29]
 [2367   37]]

Model         : Random Forest
Test Accuracy (40% hold-out): 0.5512  (55.12%)
              precision    recall  f1-score   support

   Down/Flat       0.56      0.94      0.70      3031
          Up       0.44      0.06      0.10      2404

    accuracy                           0.55      5435
   macro avg       0.50      0.50      0.40      5435
weighted avg       0.51      0.55      0.43      5435

Confusion Matrix:
[[2863  168]
 [2271  133]]

✅ Best Model Selected : Logistic Regression  (Acc = 0.5592)


### 4.3  Peer Analysis

In [25]:
# Latest snapshot per ticker
latest = df.sort_values('date').groupby('Ticker').last().reset_index()

PEER_METRICS = ['growth_proxy','valuation_proxy','institutional_flow','rsi_14','vol_ratio']
peer_medians = latest[PEER_METRICS].median()

peer_df = latest[['Ticker'] + PEER_METRICS].copy()
for m in PEER_METRICS:
    peer_df[f'{m}_vs_peer'] = (peer_df[m] - peer_medians[m]).round(6)
    peer_df[f'{m}_rank']    = peer_df[m].rank(ascending=False)

print('Group Medians:')
print(peer_medians.to_string())
print('\nPeer Analysis (latest snapshot):')
peer_df


Group Medians:
growth_proxy          0.0295
valuation_proxy      -0.0749
institutional_flow    0.0000
rsi_14               51.6457
vol_ratio             0.6108

Peer Analysis (latest snapshot):


,Ticker,growth_proxy,valuation_proxy,institutional_flow,rsi_14,vol_ratio,growth_proxy_vs_peer,growth_proxy_rank,valuation_proxy_vs_peer,valuation_proxy_rank,institutional_flow_vs_peer,institutional_flow_rank,rsi_14_vs_peer,rsi_14_rank,vol_ratio_vs_peer,vol_ratio_rank
0,ANTM,0.0326,-0.1045,0.0000,48.5294,0.8802,0.0031,3.0000,-0.0297,4.0000,0.0000,3.5000,-3.1162,4.0000,0.2694,2.0000
1,BBCA,-0.0040,-0.0452,0.0000,47.6037,0.4282,-0.0335,6.0000,0.0297,3.0000,0.0000,3.5000,-4.0419,6.0000,-0.1826,5.0000
2,BBRI,0.0385,-0.1058,0.0000,55.0000,0.5358,0.0090,2.0000,-0.0309,5.0000,0.0000,3.5000,3.3543,2.0000,-0.0750,4.0000
3,BMRI,0.0264,0.0070,0.0000,56.2500,0.6858,-0.0031,4.0000,0.0818,1.0000,0.0000,3.5000,4.6043,1.0000,0.0750,3.0000
4,BUMI,0.1160,-0.2024,0.0000,54.7619,0.9481,0.0865,1.0000,-0.1275,6.0000,0.0000,3.5000,3.1162,3.0000,0.3373,1.0000
5,TLKM,-0.0038,-0.0116,0.0000,48.1481,0.3790,-0.0333,5.0000,0.0632,2.0000,0.0000,3.5000,-3.4975,5.0000,-0.2318,6.0000


### 4.4  What Changed? — Period-over-Period Divergence

In [26]:
CHANGE_COLS = ['growth_proxy','valuation_proxy','institutional_flow','forecast_proxy','rsi_14']
wc_records  = []
for ticker, grp in df.groupby('Ticker'):
    grp = grp.sort_values('date')
    if len(grp) < 21:
        continue
    curr  = grp.iloc[-1]
    prior = grp.iloc[-21]   # ~1 calendar month
    rec   = {'Ticker': ticker}
    for col in CHANGE_COLS:
        d = curr[col] - prior[col]
        rec[f'{col}_delta'] = round(d, 6)
        rec[f'{col}_dir']   = '\u2191' if d > 0 else ('\u2193' if d < 0 else '\u2192')
    wc_records.append(rec)

wc_df = pd.DataFrame(wc_records)
print('What Changed (last 20 trading days):')
wc_df


What Changed (last 20 trading days):


,Ticker,growth_proxy_delta,growth_proxy_dir,valuation_proxy_delta,valuation_proxy_dir,institutional_flow_delta,institutional_flow_dir,forecast_proxy_delta,forecast_proxy_dir,rsi_14_delta,rsi_14_dir
0,ANTM,0.0326,↑,-0.0417,↓,0.0000,→,0.0251,↑,-8.6134,↓
1,BBCA,-0.0281,↓,0.0262,↑,0.0000,→,-0.0211,↓,-7.3963,↓
2,BBRI,-0.0120,↓,-0.0795,↓,0.0000,→,-0.0233,↓,-13.0851,↓
3,BMRI,0.0956,↑,-0.0072,↓,0.0000,→,-0.1041,↓,5.3409,↑
4,BUMI,-0.0906,↓,-0.1561,↓,0.0000,→,0.0409,↑,-3.6996,↓
5,TLKM,0.0112,↑,-0.0976,↓,0.0000,→,0.0220,↑,-4.0258,↓


### 4.5  Anomaly Detection (IsolationForest per-ticker)

In [27]:
ANOMALY_FEATURES = ['return_1d','vol_ratio','intraday_range','vol_20d']
anom_records     = []

for ticker in df['Ticker'].unique():
    tr = train_df[train_df['Ticker'] == ticker][ANOMALY_FEATURES].dropna()
    al = df[df['Ticker'] == ticker].sort_values('date').copy()
    if len(tr) < 30:
        print(f'  {ticker}: not enough train rows ({len(tr)}) — skipped')
        continue
    iso = IsolationForest(n_estimators=100, contamination=0.05, random_state=RANDOM_STATE)
    iso.fit(tr)
    X_al = al[ANOMALY_FEATURES].fillna(0)
    al['anomaly_score'] = iso.decision_function(X_al)
    al['is_anomaly']    = al['anomaly_score'] < -0.05  # threshold lebih sensitif
    lr = al.iloc[-1]
    anom_records.append({
        'Ticker'       : ticker,
        'date'         : lr['date'],
        'anomaly_score': round(float(lr['anomaly_score']), 4),
        'is_anomaly'   : bool(lr['is_anomaly']),
        'return_1d'    : round(float(lr['return_1d']), 4),
        'vol_ratio'    : round(float(lr['vol_ratio']), 4),
    })

anomaly_df = pd.DataFrame(anom_records)
print('Anomaly Detection Results (latest per ticker):')
anomaly_df


Anomaly Detection Results (latest per ticker):


,Ticker,date,anomaly_score,is_anomaly,return_1d,vol_ratio
0,ANTM,2026-09-14,0.1737,False,-0.0306,0.8802
1,BBCA,2026-09-14,0.1951,False,0.0000,0.4282
2,BBRI,2026-09-14,0.2020,False,-0.0092,0.5358
3,BMRI,2026-09-14,0.1658,False,-0.0183,0.6858
4,BUMI,2026-09-14,0.1656,False,-0.0472,0.9481
5,TLKM,2026-09-14,0.1086,False,0.0038,0.3790


### 4.6  Fundamental Divergence Detection

In [28]:
div_records = []
for ticker, grp in df.groupby('Ticker'):
    grp = grp.sort_values('date')
    if len(grp) < 21:
        continue
    curr  = grp.iloc[-1]
    prior = grp.iloc[-21]
    g_up = curr['growth_proxy']       > prior['growth_proxy']
    f_up = curr['forecast_proxy']     > prior['forecast_proxy']
    i_up = curr['institutional_flow'] > prior['institutional_flow']
    v_dn = curr['valuation_proxy']    < prior['valuation_proxy']

    sup = []
    if g_up: sup.append('Revenue/Growth improved (\u2191)')
    if f_up: sup.append('Future Forecast improved (\u2191)')
    if i_up: sup.append('Institutional Flow increased (\u2191)')
    if v_dn: sup.append('Valuation compressed vs prior (\u2193 = cheaper)')

    detected  = (g_up or f_up or i_up) and v_dn
    up_cnt    = sum([g_up, f_up, i_up, v_dn])
    conf      = 'High' if up_cnt >= 3 else ('Medium' if up_cnt == 2 else 'Low')
    div_records.append({'Ticker':ticker,'detected':detected,'confidence':conf,'supporting_factors':sup})

divergence_df = pd.DataFrame(div_records)
print('Fundamental Divergence:')
divergence_df[['Ticker','detected','confidence']]


Fundamental Divergence:


,Ticker,detected,confidence
0,ANTM,True,High
1,BBCA,False,Low
2,BBRI,False,Low
3,BMRI,True,Medium
4,BUMI,True,Medium
5,TLKM,True,High


### 4.7  Opportunity & Risk Scoring

In [29]:
# Weights per plan: Growth(25%) Valuation(25%) Institutional(20%) Forecast(20%) Peer(10%)
SCORE_FEATURES = ['growth_proxy','valuation_proxy','institutional_flow','forecast_proxy']
WEIGHTS = {
    'growth_proxy'      : 0.25,
    'valuation_proxy'   : 0.25,
    'institutional_flow': 0.20,
    'forecast_proxy'    : 0.20,
    'peer_position'     : 0.10,
}

# ── Snapshot latest per ticker
ls = latest.copy()

# ── Enrich snapshot `ls` dengan real API values (peer-relative, hanya 6 baris)
#    Proxy historis di df dibiarkan utuh untuk ML training
for ticker in TICKERS:
    f       = fundamental.get(ticker, {})
    rmask   = ls['Ticker'] == ticker
    if not rmask.any():
        continue
    close_v = float(ls.loc[rmask, 'close'].iloc[0])

    # 1) valuation_proxy: margin of safety (intrinsic vs harga pasar)
    if f.get('intrinsic_value') and f['intrinsic_value'] > 0:
        ls.loc[rmask, 'valuation_proxy'] = f['intrinsic_value'] / close_v - 1

    # 2) growth_proxy: EPS growth nyata
    if f.get('eps_growth') is not None:
        ls.loc[rmask, 'growth_proxy'] = float(f['eps_growth'])

    # 3) forecast_proxy: analyst upside vs harga terkini
    if f.get('target_mean') and f['target_mean'] > 0:
        ls.loc[rmask, 'forecast_proxy'] = f['target_mean'] / close_v - 1

    # 4) institutional_flow: top holder pct (skala ke %, agar sebanding vol_ratio)
    if f.get('inst_top_pct') is not None:
        ls.loc[rmask, 'institutional_flow'] = float(f['inst_top_pct']) * 100

# ── Peer-relative scoring: fit scaler PADA `ls` sendiri (6 tickers)
#    Ini membuat z-score bermakna: "di atas / di bawah rata-rata peers"
score_scaler = StandardScaler()
score_scaler.fit(ls[SCORE_FEATURES].fillna(0))

z_mat = score_scaler.transform(ls[SCORE_FEATURES].fillna(0))
z_df  = pd.DataFrame(z_mat, columns=SCORE_FEATURES, index=ls.index)
z_df['peer_position'] = (ls['growth_proxy'].rank() / len(ls) - 0.5) * 2  # normalised -1..+1

# ── Diagnostic: tampilkan z-scores dan raw values
print('Raw API feature values (latest per ticker):')
print(ls[['Ticker'] + SCORE_FEATURES].to_string())
print('\nZ-scores (peer-relative):')
diag = z_df.copy()
diag.insert(0, 'Ticker', ls['Ticker'].values)
print(diag.to_string())

# ── Hitung opportunity score
ls['opp_score'] = 50.0
for feat, w in WEIGHTS.items():
    ls['opp_score'] += z_df[feat] * w * 25     # each unit Z contributes proportionally

ls['opp_score']  = ls['opp_score'].clip(1, 99).round(1)
ls['direction']  = np.where(ls['opp_score'] > 50, 'Bullish', 'Bearish')
ls['risk']       = pd.cut(ls['vol_20d'],
                          bins=[0, 0.01, 0.02, np.inf],
                          labels=['Low','Medium','High'])
dist = (ls['opp_score'] - 50).abs()
ls['confidence'] = pd.cut(dist, bins=[0, 10, 20, np.inf], labels=['Low','Medium','High'])

latest_scored = ls
print('\nOpportunity & Risk Scores:')
latest_scored[['Ticker','opp_score','direction','confidence','risk']]


Raw API feature values (latest per ticker):
  Ticker  growth_proxy  valuation_proxy  institutional_flow  forecast_proxy
0   ANTM        0.0326          -0.1045              0.0000         -0.0034
1   BBCA       -0.0040          -0.0452              0.0000         -0.0413
2   BBRI        0.0385          -0.1058              0.0000         -0.0770
3   BMRI        0.0264           0.0070              0.0000         -0.0514
4   BUMI        0.1160          -0.2024              0.0000         -0.1978
5   TLKM       -0.0038          -0.0116              0.0000          0.0038

Z-scores (peer-relative):
  Ticker  growth_proxy  valuation_proxy  institutional_flow  forecast_proxy  peer_position
0   ANTM       -0.0422          -0.3903              0.0000          0.8624         0.3333
1   BBCA       -0.9527           0.4539              0.0000          0.2968        -0.6667
2   BBRI        0.1043          -0.4084              0.0000         -0.2363         0.6667
3   BMRI       -0.1965           

,Ticker,opp_score,direction,confidence,risk
0,ANTM,52.4000,Bullish,Low,Medium
1,BBCA,46.7000,Bearish,Low,Medium
2,BBRI,48.6000,Bearish,Low,Medium
3,BMRI,57.0000,Bullish,Low,Medium
4,BUMI,43.9000,Bearish,Low,High
5,TLKM,53.9000,Bullish,Low,Low


### 4.8  Evidence Generation

In [30]:
ev_records = []
for _, row in latest_scored.iterrows():
    ticker = row['Ticker']
    pos, neg, ev = [], [], []

    # Growth
    gp_c = round(row['growth_proxy']*100, 2)
    gp_p = round(peer_medians['growth_proxy']*100, 2)
    if row['growth_proxy'] > peer_medians['growth_proxy']:
        pos.append('Growth above peer median')
    else:
        neg.append('Growth below peer median')
    ev.append(f'Growth: {gp_c}% vs peer {gp_p}%')

    # Valuation
    vp_c = round(row['valuation_proxy']*100, 2)
    vp_p = round(peer_medians['valuation_proxy']*100, 2)
    if row['valuation_proxy'] > peer_medians['valuation_proxy']:
        pos.append('Valuation attractive vs peer (\u2193 60d momentum = cheaper)')
    else:
        neg.append('Valuation stretched vs peer')
    ev.append(f'Valuation proxy: {vp_c} vs peer {vp_p}')

    # Institutional flow
    if_c = round(row['institutional_flow'], 2)
    if_p = round(peer_medians['institutional_flow'], 2)
    if row['institutional_flow'] > peer_medians['institutional_flow']:
        pos.append('Institutional flow above peer')
    else:
        neg.append('Institutional flow below peer')
    ev.append(f'Institutional Flow: {if_c}x vs peer {if_p}x')

    # RSI
    rsi = round(row['rsi_14'], 1)
    if rsi > 70:
        neg.append(f'RSI overbought ({rsi})')
        ev.append(f'RSI-14: {rsi} (OB threshold = 70)')
    elif rsi < 30:
        pos.append(f'RSI oversold — potential reversal ({rsi})')
        ev.append(f'RSI-14: {rsi} (OS threshold = 30)')
    else:
        ev.append(f'RSI-14: {rsi} (Neutral zone)')

    # Dividends
    div = round(row['div_yield']*100, 4)
    if row['dividends'] > 0:
        pos.append(f'Active dividend payer (yield = {div}%)')
    else:
        neg.append('No recent dividend payment')

    ev_records.append({'Ticker':ticker,'positive_factors':pos,'negative_factors':neg,'evidence':ev})

evidence_df = pd.DataFrame(ev_records)
print('Evidence Generation complete.')
evidence_df[['Ticker','positive_factors','negative_factors']]


Evidence Generation complete.


,Ticker,positive_factors,negative_factors
0,ANTM,[Growth above peer median],"[Valuation stretched vs peer, Institutional fl..."
1,BBCA,[Valuation attractive vs peer (↓ 60d momentum ...,"[Growth below peer median, Institutional flow ..."
2,BBRI,[Growth above peer median],"[Valuation stretched vs peer, Institutional fl..."
3,BMRI,[Valuation attractive vs peer (↓ 60d momentum ...,"[Growth below peer median, Institutional flow ..."
4,BUMI,[Growth above peer median],"[Valuation stretched vs peer, Institutional fl..."
5,TLKM,[Valuation attractive vs peer (↓ 60d momentum ...,"[Growth below peer median, Institutional flow ..."


## 5. Structured JSON Output

In [31]:
timestamp = datetime.utcnow().strftime('%Y-%m-%dT%H:%M:%S')
output    = []

for ticker in sorted(df['Ticker'].unique()):
    sr = latest_scored[latest_scored['Ticker'] == ticker]
    ar = anomaly_df[anomaly_df['Ticker'] == ticker]
    dr = divergence_df[divergence_df['Ticker'] == ticker]
    er = evidence_df[evidence_df['Ticker'] == ticker]
    if sr.empty:
        continue
    s = sr.iloc[0]
    a = ar.iloc[0] if not ar.empty else {}
    d = dr.iloc[0] if not dr.empty else {}
    e = er.iloc[0] if not er.empty else {}

    record = {
        'ticker': ticker,
        'intelligence_output': {
            'anomaly'         : bool(a.get('is_anomaly', False)),
            'direction'       : str(s['direction']),
            'score'           : float(s['opp_score']),
            'confidence'      : str(s['confidence']),
            'risk'            : str(s['risk']),
            'positive_factors': list(e.get('positive_factors', [])),
            'negative_factors': list(e.get('negative_factors', [])),
        },
        'fundamental_divergence': {
            'detected'          : bool(d.get('detected', False)),
            'confidence'        : str(d.get('confidence', 'Low')),
            'supporting_factors': list(d.get('supporting_factors', [])),
        },
        'evidence' : list(e.get('evidence', [])),
        'timestamp': timestamp,
    }
    output.append(record)

print(json.dumps(output, indent=2, ensure_ascii=False))


[
  {
    "ticker": "ANTM",
    "intelligence_output": {
      "anomaly": false,
      "direction": "Bullish",
      "score": 52.4,
      "confidence": "Low",
      "risk": "Medium",
      "positive_factors": [
        "Growth above peer median"
      ],
      "negative_factors": [
        "Valuation stretched vs peer",
        "Institutional flow below peer",
        "No recent dividend payment"
      ]
    },
    "fundamental_divergence": {
      "detected": true,
      "confidence": "High",
      "supporting_factors": [
        "Revenue/Growth improved (↑)",
        "Future Forecast improved (↑)",
        "Valuation compressed vs prior (↓ = cheaper)"
      ]
    },
    "evidence": [
      "Growth: 3.26% vs peer 2.95%",
      "Valuation proxy: -10.45 vs peer -7.49",
      "Institutional Flow: 0.0x vs peer 0.0x",
      "RSI-14: 48.5 (Neutral zone)"
    ],
    "timestamp": "2026-09-14T07:07:00"
  },
  {
    "ticker": "BBCA",
    "intelligence_output": {
      "anomaly": false,
      "d

## 6. Model Accuracy Summary (60% Train / 40% Test)

In [32]:
print('=' * 60)
print('MODEL ACCURACY SUMMARY — Temporal 60/40 Split')
print('=' * 60)
for name, res in results.items():
    marker = '<-- BEST' if name == best_name else ''
    print(f"  {name:<30} Acc: {res['acc']:.4f}  ({res['acc']*100:.2f}%)  {marker}")
print()
print(f'  Best Model : {best_name}')
print(f'  Best Acc   : {results[best_name]["acc"]:.4f}')
print('=' * 60)
print('\nFinal Signal Summary per Ticker:')
for rec in output:
    io = rec['intelligence_output']
    fd = rec['fundamental_divergence']
    print(
        f"  [{rec['ticker']}]  Score={io['score']:<5}  Dir={io['direction']:<8} "
        f"Conf={io['confidence']:<7} Risk={io['risk']:<7} "
        f"Anomaly={io['anomaly']}  FundDiv={fd['detected']}"
    )


MODEL ACCURACY SUMMARY — Temporal 60/40 Split
  Logistic Regression            Acc: 0.5592  (55.92%)  <-- BEST
  Random Forest                  Acc: 0.5512  (55.12%)  

  Best Model : Logistic Regression
  Best Acc   : 0.5592

Final Signal Summary per Ticker:
  [ANTM]  Score=52.4   Dir=Bullish  Conf=Low     Risk=Medium  Anomaly=False  FundDiv=True
  [BBCA]  Score=46.7   Dir=Bearish  Conf=Low     Risk=Medium  Anomaly=False  FundDiv=False
  [BBRI]  Score=48.6   Dir=Bearish  Conf=Low     Risk=Medium  Anomaly=False  FundDiv=False
  [BMRI]  Score=57.0   Dir=Bullish  Conf=Low     Risk=Medium  Anomaly=False  FundDiv=True
  [BUMI]  Score=43.9   Dir=Bearish  Conf=Low     Risk=High    Anomaly=False  FundDiv=True
  [TLKM]  Score=53.9   Dir=Bullish  Conf=Low     Risk=Low     Anomaly=False  FundDiv=True
